# Transformada Cuántica de Fourier (QFT)

**Objetivo:** construir la QFT de 3 qubits desde cero y verificar su unitariedad.

La QFT es el núcleo de QPE, Shor y HHL. Transforma la base computacional a la base de frecuencias:
|j⟩ → (1/√N) Σ_k e^(2πijk/N) |k⟩

In [ ]:
import numpy as np

# DFT clásica para comparar
N = 8  # 3 qubits → 2^3 = 8 elementos
j = np.arange(N)

# Señal de ejemplo
x = np.array([1, 0, 0, 1, 0, 0, 1, 0], dtype=complex)
x_dft = np.fft.fft(x) / np.sqrt(N)  # DFT normalizada
print('Señal original:', x)
print('DFT normalizada:', np.round(x_dft, 3))

## Matriz QFT

La QFT es la matriz DFT unitaria: QFT_jk = (1/√N) e^(2πijk/N).
Verifiquemos que es unitaria (QFT† × QFT = I):

In [ ]:
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    j, k = np.meshgrid(np.arange(N), np.arange(N))
    return omega**(j*k) / np.sqrt(N)

QFT3 = qft_matrix(3)
print('Unitaria:', np.allclose(QFT3.conj().T @ QFT3, np.eye(8)))
print('Dimensión:', QFT3.shape)

## Circuito QFT

La QFT se implementa con puertas H y R_k (rotación de fase de 2π/2^k):

```
q0: ─H─R2─R3─────────────SWAP─
q1: ────●──│──H──R2──────────●─
q2: ───────●─────●───H───────  
```

In [ ]:
from qiskit import QuantumCircuit
import numpy as np

def build_qft(n):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
        for j in range(i+1, n):
            angle = 2 * np.pi / 2**(j - i + 1)
            qc.cp(angle, j, i)   # R_k controlada
    # SWAPs finales para invertir el orden
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    return qc

qft3 = build_qft(3)
print(qft3.draw())

In [ ]:
# Verificar que produce la misma transformación que la matriz
from qiskit.quantum_info import Operator

U_qiskit = Operator(qft3).data
print('Coincide con matriz analítica:', np.allclose(U_qiskit, QFT3, atol=1e-10))
print('Unitaria:', np.allclose(U_qiskit @ U_qiskit.conj().T, np.eye(8), atol=1e-10))

In [ ]:
# Aplicar QFT a un estado con período 2
from qiskit.quantum_info import Statevector

# |x⟩ = (|000⟩ + |010⟩ + |100⟩ + |110⟩)/2 — período 2
psi = np.zeros(8); psi[[0,2,4,6]] = 0.5
sv = Statevector(psi)

result = sv.evolve(qft3)
print('Estado con período 2 — amplitudes QFT:')
for i, amp in enumerate(result.data):
    if abs(amp) > 0.1:
        print(f'  |{format(i,"03b")}⟩: {amp:.3f}  (prob={abs(amp)**2:.3f})')

## Comparativa: QFT de Qiskit vs manual

In [ ]:
from qiskit.circuit.library import QFT

qft_lib = QFT(3, do_swaps=True)
U_lib = Operator(qft_lib).data
print('QFT manual == QFT Qiskit:', np.allclose(U_qiskit, U_lib, atol=1e-10))

## Ejercicio

1. Verifica que QFT⁻¹ = QFT† (inversa es la adjunta).
2. Aplica la QFT al estado |001⟩ y analiza el resultado.
3. ¿Por qué la QFT necesita los SWAPs al final?

In [ ]:
# Tu solución
U_inv = U_qiskit.conj().T
print('QFT⁻¹ = QFT†:', np.allclose(U_inv @ U_qiskit, np.eye(8)))

psi_001 = np.zeros(8); psi_001[1] = 1.0
result_001 = U_qiskit @ psi_001
print('QFT|001⟩:', np.round(result_001, 3))

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/05_qft_tres_qubits.ipynb`
- **Módulo:** `Tutorial/05_algoritmos/README.md`
- **Siguiente guía:** `06_estimacion_de_fase.ipynb`